In [2]:
import pandas as pd

df1 = pd.read_csv("Dataset1.csv")
df2 = pd.read_csv("Dataset2.csv")
df3 = pd.read_csv("Dataset4.csv")

print("Dataset1 Shape:", df1.shape)
print("Dataset2 Shape:", df2.shape)
print("Dataset4 Shape:", df3.shape)

Dataset1 Shape: (2200, 8)
Dataset2 Shape: (100000, 8)
Dataset4 Shape: (20000, 8)


In [3]:
import pandas as pd

# Merge
df = pd.concat([df1, df2, df3], ignore_index=True)

print("After Merge Shape:", df.shape)

# Clean labels (VERY IMPORTANT)
df['label'] = df['label'].str.lower().str.strip()

print("\nUnique Labels:", df['label'].nunique())
print(df['label'].value_counts().head())

After Merge Shape: (122200, 8)

Unique Labels: 33
label
sugarcane    12551
maize        12465
wheat        12302
cotton        9337
tobacco       9224
Name: count, dtype: int64


In [4]:
counts = df['label'].value_counts()

print(counts.tail(10))   # sabse chhoti classes dikhao

label
banana         100
pomegranate    100
lentil         100
blackgram      100
mungbean       100
mothbeans      100
coffee         100
papaya         100
coconut        100
jute           100
Name: count, dtype: int64


In [21]:
counts = df['label'].value_counts()

# Keep only labels having >= 500 samples
valid_labels = counts[counts >= 500].index

df = df[df['label'].isin(valid_labels)]

print("New Shape:", df.shape)
print("Remaining Classes:", df['label'].nunique())
print(df['label'].value_counts().head(14))

New Shape: (120300, 14)
Remaining Classes: 14
label
sugarcane      12551
maize          12465
wheat          12302
cotton          9337
tobacco         9224
millets         9154
paddy           9103
oil seeds       9096
pulses          9072
barley          9041
ground nuts     8881
rice            3368
potato          3362
tomato          3344
Name: count, dtype: int64


In [6]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['label'])

print("Total Classes:", len(le.classes_))
print(df[['label', 'label_encoded']].head())

Total Classes: 14
  label  label_encoded
0  rice              9
1  rice              9
2  rice              9
3  rice              9
4  rice              9


In [7]:
# Features & Target
X = df.drop(['label', 'label_encoded'], axis=1)
y = df['label_encoded']

print("Feature Shape:", X.shape)
print("Target Shape:", y.shape)

Feature Shape: (120300, 7)
Target Shape: (120300,)


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train Shape:", X_train.shape)
print("Test Shape:", X_test.shape)

Train Shape: (96240, 7)
Test Shape: (24060, 7)


In [9]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softmax',
    num_class=14,
    random_state=42,
    eval_metric='mlogloss'
)

model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softmax'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes fro

In [10]:
from sklearn.metrics import accuracy_score

# Predictions
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# Accuracy
train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print("Train Accuracy:", train_acc)
print("Test Accuracy:", test_acc)

Train Accuracy: 0.6358270989193683
Test Accuracy: 0.4150041562759767


In [11]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_test_pred))

              precision    recall  f1-score   support

           0       0.25      0.04      0.06      1808
           1       0.65      0.85      0.74      1867
           2       0.35      0.14      0.20      1776
           3       0.23      0.48      0.31      2493
           4       0.36      0.75      0.49      1831
           5       0.45      0.38      0.41      1819
           6       0.67      0.88      0.76      1821
           7       0.19      0.19      0.19       673
           8       0.26      0.07      0.11      1814
           9       0.18      0.20      0.19       674
          10       0.69      0.50      0.58      2510
          11       0.51      0.74      0.60      1845
          12       0.18      0.18      0.18       669
          13       0.16      0.05      0.08      2460

    accuracy                           0.42     24060
   macro avg       0.37      0.39      0.35     24060
weighted avg       0.39      0.42      0.37     24060



In [12]:
df['npk_total'] = df['N'] + df['P'] + df['K']

In [13]:
df['n_p_ratio'] = df['N'] / (df['P'] + 1)
df['n_k_ratio'] = df['N'] / (df['K'] + 1)
df['p_k_ratio'] = df['P'] / (df['K'] + 1)

In [14]:
df['temp_humidity'] = df['temperature'] * df['humidity']

In [15]:
df['npk_total'] = df['N'] + df['P'] + df['K']
df['n_p_ratio'] = df['N'] / (df['P'] + 1)
df['n_k_ratio'] = df['N'] / (df['K'] + 1)
df['p_k_ratio'] = df['P'] / (df['K'] + 1)
df['temp_humidity'] = df['temperature'] * df['humidity']

print(df.head())

      N     P     K  temperature   humidity        ph    rainfall label  \
0  90.0  42.0  43.0    20.879744  82.002744  6.502985  202.935536  rice   
1  85.0  58.0  41.0    21.770462  80.319644  7.038096  226.655537  rice   
2  60.0  55.0  44.0    23.004459  82.320763  7.840207  263.964248  rice   
3  74.0  35.0  40.0    26.491096  80.158363  6.980401  242.864034  rice   
4  78.0  42.0  42.0    20.130175  81.604873  7.628473  262.717340  rice   

   label_encoded  npk_total  n_p_ratio  n_k_ratio  p_k_ratio  temp_humidity  
0              9      175.0   2.093023   2.045455   0.954545    1712.196283  
1              9      184.0   1.440678   2.023810   1.380952    1748.595734  
2              9      159.0   1.071429   1.333333   1.222222    1893.744627  
3              9      149.0   2.055556   1.804878   0.853659    2123.482908  
4              9      162.0   1.813953   1.813953   0.976744    1642.720357  


In [16]:
# Redefine Features (now including new engineered features)
X = df.drop(['label', 'label_encoded'], axis=1)
y = df['label_encoded']

print("New Feature Shape:", X.shape)

New Feature Shape: (120300, 12)


In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train Shape:", X_train.shape)
print("Test Shape:", X_test.shape)

Train Shape: (96240, 12)
Test Shape: (24060, 12)


In [19]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=400,
    max_depth=7,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',
    num_class=14,
    random_state=42,
    eval_metric='mlogloss'
)

model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes fr

In [20]:
import numpy as np
from sklearn.metrics import top_k_accuracy_score

# Get probability predictions
y_proba = model.predict_proba(X_test)

# Top-3 Accuracy
top3_acc = top_k_accuracy_score(y_test, y_proba, k=3)

print("Top-1 Accuracy:", model.score(X_test, y_test))
print("Top-3 Accuracy:", top3_acc)

Top-1 Accuracy: 0.407024106400665
Top-3 Accuracy: 0.7541978387364922


In [22]:
# Create new merged label column

def merge_crops(label):
    if label in ['rice', 'paddy', 'maize', 'millets', 'barley', 'wheat']:
        return 'cereals'
    elif label in ['pulses', 'mungbean', 'lentil', 'blackgram']:
        return 'pulses_group'
    elif label in ['oil seeds', 'ground nuts']:
        return 'oil_crops'
    elif label in ['tomato', 'potato']:
        return 'vegetables'
    elif label in ['sugarcane', 'tobacco']:
        return 'cash_crops'
    else:
        return label  # cotton etc

df['merged_label'] = df['label'].apply(merge_crops)

print("New Class Count:", df['merged_label'].nunique())
print(df['merged_label'].value_counts())

New Class Count: 6
merged_label
cereals         55433
cash_crops      21775
oil_crops       17977
cotton           9337
pulses_group     9072
vegetables       6706
Name: count, dtype: int64


In [23]:
from sklearn.preprocessing import LabelEncoder

le_merged = LabelEncoder()
df['merged_encoded'] = le_merged.fit_transform(df['merged_label'])

print("Merged Classes:", le_merged.classes_)
print("Total Classes:", len(le_merged.classes_))

Merged Classes: ['cash_crops' 'cereals' 'cotton' 'oil_crops' 'pulses_group' 'vegetables']
Total Classes: 6


In [24]:
# Features
X = df.drop(['label', 'label_encoded', 'merged_label', 'merged_encoded'], axis=1)

# Target
y = df['merged_encoded']

print("Feature Shape:", X.shape)
print("Target Shape:", y.shape)

Feature Shape: (120300, 12)
Target Shape: (120300,)


In [25]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train Shape:", X_train.shape)
print("Test Shape:", X_test.shape)

Train Shape: (96240, 12)
Test Shape: (24060, 12)


In [26]:
from xgboost import XGBClassifier

model_merged = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',   # IMPORTANT
    num_class=6,
    random_state=42,
    eval_metric='mlogloss'
)

model_merged.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes fr

In [27]:
import numpy as np
from sklearn.metrics import top_k_accuracy_score

# Probability predictions
y_proba = model_merged.predict_proba(X_test)

# Top-3 accuracy
top3_acc = top_k_accuracy_score(y_test, y_proba, k=3)

print("Top-1 Accuracy:", model_merged.score(X_test, y_test))
print("Top-3 Accuracy:", top3_acc)

Top-1 Accuracy: 0.6088528678304239
Top-3 Accuracy: 0.9720282626766418
